## Sentiment Analysis
Sentiment Analysis is a basically a classification task that classify the customer sentiment from the customer feedback text.
We will perform Sentinment Analysis using
- 1. Rule-based Analysis
- 2. ML Algorithm (NB, Random Forest, and SVM)
- 3. Deep Learning (RNN and LSTM)
- 4. Pre-trained Transformers (BERT, RoBERTa, etc..)
- 5. Aspect Based Sentiment Analysis (ABSA)

In [ ]:
#!pip install -U nltk spacy langdetect ftfy contractions  emoji Tokenizer gensim scikeras scikit-learn

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Load dataset
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/edurekaai/_data/yelp.csv", encoding="latin1")

# Target
## Map stars to classes
df["sentiment"] = df["stars"].map(lambda x: "negative" if x < 3 else ("neutral" if x == 3 else "positive"))

train_data = df.copy()
# train_data=pd.read_csv("/content/drive/MyDrive/edurekaai/_data/sentiment_analysis_train.csv",encoding='latin1')
# test_data=pd.read_csv("/content/drive/MyDrive/edurekaai/_data/sentiment_analysis_test.csv",encoding='latin1')

df.head()

## 1. Lexicon + Rule-based Sentiment Analysis
It uses a dictionary of words mapped to sentiment scores. The algorithm looks up word in the text and sum up the scores. It then uses the score the classify the sentiment as Postive, Negative or Nutral.

Here are some popular dictionaries:
  - VADER is for social media reviews
  - SentiWordNet for news

How it detects sentiment:
- looks for word in the dictionary and its score
- looks for punctuations in the dictionary and its score
- looks for Capitalization
- looks for negation and its score

And then it adds up all the score. Positive score postivie, Negative score is negative, Zero is Neutral.


### VEDAR Approach

In [ ]:
import nltk
nltk.download('vader_lexicon')

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# sample_reviews = [
#     "The place was fantastic! Loved the vibe and the food.",
#     "I hated the food. It was bland and overpriced.",
#     "It was okay, nothing special.",
#     "Fast service, clean environment, and tasty meals!",
#     "Not worth the price. Wouldn't recommend."
# ]

# NLP Pre-processing steps
## 1. lowercasing - Is not needed. Becasue VADER’s lexicon is case-sensitive — it uses capitalization as a signal of emphasis.
## 2. punctuation - No explicit handling is needed. Because VADER already handles punctuation. Exclamation marks add emphasis:
## 3. tokenization - No required. VADER works directly on raw strings. It internally tokenizes as needed.
## 4. stopwords - Not required. VADER’s lexicon already includes common inflections (happy, happier, happiest).
## 5. Emoji handling - Partially handled. VADER recognizes many common emojis and emoticons.
## 6. Negation handling - Already built-in. VADER handles negations.

#Get sentiment scores for 5 sentences only
import random

# Pick 5 random rows from df["text"]
sampled_texts = df["text"].sample(n=5, random_state=42)  # random_state for reproducibility

for text in sampled_texts:
    sentiment_scores = sia.polarity_scores(text)
    print(f"Text: {text}")
    print(f"Sentiment Scores: {sentiment_scores}")
    print("---------------------------------------\n")

### Limitations with Rule-based
- It works for clean text
- Not for multi-modal messages

## 2. ML Algorithms Sentiment-Based Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re

#nltk Lib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#ML Lib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

#Download nltk resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

### 2.1 Preprocessing

In [ ]:
#Text Preprocessing: Clean URLs
def clean_text(text):
  text=str(text).lower()
  text=re.sub(r'http\S+|www\S+|https\S+','',text)
  text=re.sub(f"[{re.escape(string.punctuation)}]",'',text)
  text=re.sub(r'\d+','',text)
  return text

#Initialize stopword and lemmatizer
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

#Text Preprocessing: Tokenize,Remove Stopwords, Lemmatize and reconstruct the sentence
def preprocess_text(text):
  words=nltk.word_tokenize(text)
  words=[lemmatizer.lemmatize(word) for word in words if word not in stop_words]
  return ' '.join(words)

# Apply cleaning on training data
df['text_preprocessed']=df['text'].apply(clean_text)

# Apply Text preprocessing
df['text_preprocessed']=df['text_preprocessed'].apply(preprocess_text)


In [ ]:
# Train and Validation Split
X = df['text_preprocessed']
y = df['sentiment']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



### 2.2 EDA

In [ ]:
# unique sentiment classes
print(df['sentiment'].value_counts())

### 2.3 Representation

In [ ]:
# Embedding Vectorize the input data using TFIDF
## The vectorizer looks at all the words in your dataset → ranks them by importance (based on frequency or TF-IDF scores)
## → and keeps only the top 5000 words.
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)


### 2.4 Model Training and Evaluation

In [ ]:
#Model-1 (Naive Bays Classifier)
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_tfidf, y_train)
y_val_pred_nb = nb_classifier.predict(X_val_tfidf)

In [ ]:
#Model-2 (Random Forest Classifier)
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_tfidf, y_train)
y_val_pred_rf = rf_classifier.predict(X_val_tfidf)

In [ ]:
#Model-3 (SVM)
svm_classifier = SVC(kernel='linear', random_state=42)
svm_classifier.fit(X_train_tfidf, y_train)
y_val_pred_svm = svm_classifier.predict(X_val_tfidf)


In [ ]:
# Evaluate Models
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    classification_rep = classification_report(y_true, y_pred)
    print(f"Model: {model_name}")
    print(f"Accuracy: {accuracy}")
    print("Classification Report:")
    print(classification_rep)
    print("\n")

# Evaluate Model1
evaluate_model(y_val, y_val_pred_nb, "Naive Bayes")

# Evaluate Model2
evaluate_model(y_val, y_val_pred_rf, "Random Forest")

# Evaluate Model3
evaluate_model(y_val, y_val_pred_svm, "SVM")

### 2.5 Inferencing the Models / Deployment


In [ ]:
sample_sentiments = [
    # Positive
    "I love this product! It's amazing",
    "The food was fantastic and the service was excellent",
    "What a wonderful experience, I’ll definitely come back",
    "Fast delivery and great quality, very happy with my purchase",
    "Absolutely loved it! Exceeded my expectations",

    # Negative
    "I hated the food, it was bland and overpriced",
    "This is the worst service I’ve ever experienced",
    "The product broke after one day, very disappointed",
    "Terrible experience, I would not recommend",
    "The place was dirty and the staff were rude",

    # Neutral
    "It was okay, nothing special",
    "The movie was average, not too bad",
    "Service was fine, nothing out of the ordinary",
    "The price is reasonable for what you get",
    "The weather today is neither good nor bad"
]

# Ground truth labels for sample_sentiments
true_labels = [
    "positive", "positive", "positive", "positive", "positive",  # first 5 are positive
    "negative", "negative", "negative", "negative", "negative",  # next 5 are negative
    "neutral", "neutral", "neutral", "neutral", "neutral"       # last 5 are neutral
]

In [ ]:
from sklearn.metrics import accuracy_score

# predict sentiment using the models
def predict_sentiment(texts, model):
    preds = []
    for text in texts:
        # Step 1: clean
        text_clean = clean_text(text)
        # Step 2: preprocess (tokenize, lemmatize, remove stopwords)
        text_preprocessed = preprocess_text(text_clean)
        # Step 3: vectorize
        text_tfidf = vectorizer.transform([text_preprocessed])
        # Step 4: predict sentiment label
        sentiment = model.predict(text_tfidf)[0]
        preds.append(sentiment)
    return preds


# Evaluate each model
for model_name, model in {
    "Naive Bayes": nb_classifier,
    "Random Forest": rf_classifier,
    "SVM": svm_classifier
}.items():
    preds = predict_sentiment(sample_sentiments, model)
    acc = accuracy_score(true_labels, preds)
    print(f"\n{model_name} Predictions:")
    for sent, pred, true in zip(sample_sentiments, preds, true_labels):
        print(f"Text: {sent}\n → Predicted: {pred}, True: {true}")
    print(f"{model_name} Accuracy: {acc:.2f}")



## 3. ML Algorithms Sentiment-Based Analysis - Using Pipeline

In [ ]:
import pandas as pd
import numpy as np
import re, string
import nltk

# NLTK setup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

# Scikit-learn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# -----------------------
# Custom Preprocessor
# -----------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r"http\S+|www\S+|https\S+", "", text)      # remove URLs
        text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)  # remove punctuation
        text = re.sub(r"\d+", "", text)                         # remove digits
        return text

    def preprocess(self, text):
        words = nltk.word_tokenize(text)
        words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
        return " ".join(words)

    def transform(self, X, y=None):
        return [self.preprocess(self.clean_text(text)) for text in X]

    def fit(self, X, y=None):
        return self

# -----------------------
# Load Data
# -----------------------
df = pd.read_csv("/content/drive/MyDrive/edurekaai/_data/yelp.csv", encoding="latin1")

# -----------------------
# Target
## Map stars to classes
# -----------------------
df["sentiment"] = df["stars"].map(lambda x: "negative" if x < 3 else ("neutral" if x == 3 else "positive"))

X = df["text"]
y = df["sentiment"]

# -----------------------
# Train and Validation Split
# -----------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------
# Pipelines for Models
# -----------------------
pipelines = {
    "Naive Bayes": Pipeline([
        ("preprocess", TextPreprocessor()),
        ("vectorizer", TfidfVectorizer(max_features=5000)),
        ("classifier", MultinomialNB())
    ]),
    "Random Forest": Pipeline([
        ("preprocess", TextPreprocessor()),
        ("vectorizer", TfidfVectorizer(max_features=5000)),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "SVM": Pipeline([
        ("preprocess", TextPreprocessor()),
        ("vectorizer", TfidfVectorizer(max_features=5000)),
        ("classifier", SVC(kernel="linear", random_state=42))
    ])
}

# -----------------------
# Train & Evaluate
# -----------------------
def evaluate_pipeline(name, pipeline, X_train, y_train, X_val, y_val):
    print(f"\n=== {name} ===")
    model = pipeline.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    print(f"Accuracy: {accuracy_score(y_val, y_pred):.2f}")
    print("Classification Report:")
    print(classification_report(y_val, y_pred))
    return pipeline

fitted_pipelines = {}
for name, pipe in pipelines.items():
    fitted_pipelines[name] = evaluate_pipeline(name, pipe, X_train, y_train, X_val, y_val)

# -----------------------
# Sample Sentiment Prediction
# -----------------------
sample_sentiments = [
    "I love this product! It's amazing",
    "The food was fantastic and the service was excellent",
    "What a wonderful experience, I’ll definitely come back",
    "Fast delivery and great quality, very happy with my purchase",
    "Absolutely loved it! Exceeded my expectations",
    "I hated the food, it was bland and overpriced",
    "This is the worst service I’ve ever experienced",
    "The product broke after one day, very disappointed",
    "Terrible experience, I would not recommend",
    "The place was dirty and the staff were rude",
    "It was okay, nothing special",
    "The movie was average, not too bad",
    "Service was fine, nothing out of the ordinary",
    "The price is reasonable for what you get",
    "The weather today is neither good nor bad"
]

true_labels = [
    "positive", "positive", "positive", "positive", "positive",
    "negative", "negative", "negative", "negative", "negative",
    "neutral", "neutral", "neutral", "neutral", "neutral"
]

for name, pipe in fitted_pipelines.items():
    preds = pipe.predict(sample_sentiments)
    acc = accuracy_score(true_labels, preds)
    print(f"\n{name} Predictions:")
    for sent, pred, true in zip(sample_sentiments, preds, true_labels):
        print(f"Text: {sent}\n → Predicted: {pred}, True: {true}")
    print(f"{name} Accuracy on samples: {acc:.2f}")


## 4. Deep Learning Based Analysis
For sentiment analysis, the machine MUST process the **sequence of tokens (words in sequence)** to make the decission. Hence, we must use time-series Neural Networks (RNN, LSTM, or GRU).

### Loading

In [ ]:
# ------------------------
# 1. Import Packages
# ------------------------
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import pandas as pd
import re
import string

# ------------------------
# 2. Load Dataset
# ------------------------
df = pd.read_csv("/content/drive/MyDrive/edurekaai/_data/yelp.csv", encoding="latin1")
print(f"Dataset size: {len(df)}")
print(df.head())

### 4.1 Preprocessing

In [ ]:
# ------------------------
# 3. Clean Text Function
# ------------------------
def clean_text(text):
    """
    Cleans input text:
    - Lowercase
    - Remove URLs
    - Remove punctuation
    - Remove digits
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # remove URLs
    text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)  # remove punctuation
    text = re.sub(r'\d+', '', text)  # remove numbers
    return text

# Apply cleaning
df['text_cleaned'] = df['text'].apply(clean_text)


### 4.2 EDA
No EDA done here.

### 4.3 Representation


In [ ]:
# ------------------------
# 4. Tokenization Setup
# ------------------------
max_words = 1000   # Keep top 1000 most frequent words
max_len = 200      # Each review will be padded/truncated to 100 words

## Tokenizer.fit_on_texts() scans through your dataset’s text and builds a word-to-index dictionary (word → integer ID).
## Converts raw text into numerical sequences (so the model can understand it).
## Example:
#### Input: ["i love deep learning", "i love ai"]
#### Output: tokenizer.word_index = {'<OOV>': 1, 'i': 2, 'love': 3, 'deep': 4, 'learning': 5, 'ai': 6}
#### '<OOV>' is a placeholder used for the input word not present in the training data.
#### During the inference, for all word that was not present in the training data, it uses '<OOV>'.
## Note: fit_on_texts() does not return anything. It simplt build the dictionary tokenizer.word_index
## tokenizer.word_index dictionary is used in the tokenizer.texts_to_sequences() call
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text_cleaned'])

# Converts your sentences into lists of integers
## Replaces all word with the index# from the dictionary tokenizer.word_index
## Example1:
#### Input: ["i love deep learning", "i love ai", "i love pizza"]
#### Output: [[2, 3, 4, 5],   # "i love deep learning"
####          [2, 3, 6],      # "i love ai"
####          [2, 3, 1]]      # "i love pizza". It used index of "OOV" for 'pizza'. Because, 'pizza' was not present in the dictionary building input.
X = tokenizer.texts_to_sequences(df['text_cleaned'])

# Pads the input to make each row of same size. Because Neural networks require fixed-size input.
## Example:
#### Input: [[2, 3, 4, 5], [2, 3, 6], [2, 3, 1]]
#### Output: [[2 3 4 5 0 0]   # padded with 0s at the end
####          [2 3 6 0 0 0],  # padded with 0s at the end
####          [2 3 1 0 0 0]]  # padded with 0s at the end
## Note: pad_sequences() returns a numpy array
## padding='post' pads at the end
## truncating='post' truncates the array to max_len.
X = pad_sequences(X, maxlen=max_len, padding='post', truncating='post')

print(f"Feature shape: {X.shape}")  # (num_samples, max_len)

# ------------------------
# 5. Prepare Labels
# ------------------------
# Map Yelp stars to sentiment categories
# 1-2 stars = Negative (0), 3 = Neutral (1), 4-5 = Positive (2)
df["sentiment"] = df["stars"].map(lambda x: 0 if x < 3 else (1 if x == 3 else 2))

# One-hot encode labels for categorical crossentropy
y = to_categorical(df["sentiment"], num_classes=3)
print(f"Label shape: {y.shape}")  # (num_samples, 3)
print(X[0])
print(y[0])



### 4.4 Model Building and Evaluating

In [ ]:
# ------------------------
# 6. Train-Test Split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------
# 7. Build Model
# ------------------------
model = Sequential([
    Input(shape=(max_len,)),                                # Input Layer: It defines it of max_len size.
    Embedding(input_dim=max_words, output_dim=64),          # Embedding Layer: It vectorizes the input indices representing the actual word.
                                                            # By doing so we are actually normizing the input
                                                            # input_dim=max_words - is the size of the word dictionary generated duing the representation.
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),           # LSTM layer with dropout
    Dense(3, activation='softmax')                          # Output: 3 classes
])

model.compile(loss="categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.summary()

# ------------------------
# 8. Train Model
# ------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,   # Keep 20% of training data for validation
    epochs=10,
    batch_size=32,
    verbose=1
)

# ------------------------
# 9. Evaluate Model
# ------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Predict probabilities
y_pred_probs = model.predict(X_test)
y_pred = y_pred_probs.argmax(axis=1)   # Convert to class labels
y_true = y_test.argmax(axis=1)

# Metrics
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

### 4.5 Inferencing / Evaluating

In [ ]:
# -----------------------
# Sample Sentiment Prediction
# -----------------------
sample_sentiments = [
    "I love this product! It's amazing",
    "The food was fantastic and the service was excellent",
    "What a wonderful experience, I’ll definitely come back",
    "Fast delivery and great quality, very happy with my purchase",
    "Absolutely loved it! Exceeded my expectations",
    "I hated the food, it was bland and overpriced",
    "This is the worst service I’ve ever experienced",
    "The product broke after one day, very disappointed",
    "Terrible experience, I would not recommend",
    "The place was dirty and the staff were rude",
    "It was okay, nothing special",
    "The movie was average, not too bad",
    "Service was fine, nothing out of the ordinary",
    "The price is reasonable for what you get",
    "The weather today is neither good nor bad"
]

true_labels = [
    "positive", "positive", "positive", "positive", "positive",
    "negative", "negative", "negative", "negative", "negative",
    "neutral", "neutral", "neutral", "neutral", "neutral"
]

# Clean the text
sample_cleaned = [clean_text(text) for text in sample_sentiments]

# Convert to sequences using the same tokenizer
sample_seq = tokenizer.texts_to_sequences(sample_cleaned)

# Pad sequences to max_len
sample_padded = pad_sequences(sample_seq, maxlen=max_len, padding='post', truncating='post')

# Predict probabilities
sample_probs = model.predict(sample_padded)

# Convert to class labels (0 = negative, 1 = neutral, 2 = positive)
sample_preds = sample_probs.argmax(axis=1)


label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
sample_preds_labels = [label_map[p] for p in sample_preds]

from sklearn.metrics import classification_report, confusion_matrix

# Convert true_labels to numeric values
true_map = {'negative': 0, 'neutral': 1, 'positive': 2}
y_true_nums = [true_map[label] for label in true_labels]

print(classification_report(y_true_nums, sample_preds))
print(confusion_matrix(y_true_nums, sample_preds))



## 5. Deep Learning Based Analysis using Pipeline

In [ ]:
# ------------------------
# 1. Import Packages
# ------------------------
import re
import string
import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------
# 2. Load Dataset
# ------------------------
df = pd.read_csv("/content/drive/MyDrive/edurekaai/_data/yelp.csv", encoding="latin1")

# ------------------------
# Remove neutral reviews
# ------------------------
df = df[df['stars'] != 3].reset_index(drop=True)
print(f"Dataset size: {len(df)}")
print(df.head())

# ------------------------
# 3. Custom Transformer for Text Preprocessing
# ------------------------
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, max_words=1000, max_len=200):
        self.max_words = max_words
        self.max_len = max_len
        self.tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')

    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)
        text = re.sub(r'\d+', '', text)
        return text

    def fit(self, X, y=None):
        cleaned_texts = [self.clean_text(text) for text in X]
        self.tokenizer.fit_on_texts(cleaned_texts)
        return self

    def transform(self, X):
        cleaned_texts = [self.clean_text(text) for text in X]
        sequences = self.tokenizer.texts_to_sequences(cleaned_texts)
        padded = pad_sequences(sequences, maxlen=self.max_len, padding='post', truncating='post')
        return padded

# ------------------------
# 4. Prepare Labels
# ------------------------
# Map Yelp stars to sentiment categories
df["sentiment"] = df["stars"].map(lambda x: 0 if x < 3 else (1 if x == 3 else 2))
y = to_categorical(df["sentiment"], num_classes=3)

# ------------------------
# 5. Train-Test Split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], y, test_size=0.2, random_state=42
)

# ------------------------
# 6. Function to Build Model
# ------------------------
def build_model(max_words=1000, max_len=200):
    model = Sequential([
        Input(shape=(max_len,)),
        Embedding(input_dim=max_words, output_dim=64),
        GRU(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(3, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# ------------------------
# 7. Create Preprocessing Pipeline
# ------------------------
# Only preprocessing for now
preprocessor = TextPreprocessor(max_words=1000, max_len=200)
X_train_padded = preprocessor.fit_transform(X_train)
X_test_padded = preprocessor.transform(X_test)

# ------------------------
# 8. Train Model
# ------------------------
model = build_model(max_words=1000, max_len=200)
history = model.fit(
    X_train_padded, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1
)

# ------------------------
# 9. Evaluate Model
# ------------------------
loss, accuracy = model.evaluate(X_test_padded, y_test, verbose=0)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

y_pred_probs = model.predict(X_test_padded)
y_pred = y_pred_probs.argmax(axis=1)
y_true = y_test.argmax(axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


In [ ]:
# ------------------------
# Sample Sentiment Prediction using Pipeline
# ------------------------
sample_sentiments = [
    "I love this product! It's amazing",
    "The food was fantastic and the service was excellent",
    "What a wonderful experience, I’ll definitely come back",
    "Fast delivery and great quality, very happy with my purchase",
    "Absolutely loved it! Exceeded my expectations",
    "I hated the food, it was bland and overpriced",
    "This is the worst service I’ve ever experienced",
    "The product broke after one day, very disappointed",
    "Terrible experience, I would not recommend",
    "The place was dirty and the staff were rude",
    "It was okay, nothing special",
    "The movie was average, not too bad",
    "Service was fine, nothing out of the ordinary",
    "The price is reasonable for what you get",
    "The weather today is neither good nor bad"
]

true_labels = [
    "positive", "positive", "positive", "positive", "positive",
    "negative", "negative", "negative", "negative", "negative",
    "neutral", "neutral", "neutral", "neutral", "neutral"
]

# ------------------------
# 1. Preprocess Sample Data
# ------------------------
# Use the same preprocessor instance from training
sample_padded = preprocessor.transform(sample_sentiments)

# ------------------------
# 2. Predict Probabilities
# ------------------------
sample_probs = model.predict(sample_padded)

# ------------------------
# 3. Convert Probabilities to Class Labels
# ------------------------
sample_pred_indices = sample_probs.argmax(axis=1)
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
sample_pred_labels = [label_map[idx] for idx in sample_pred_indices]

# ------------------------
# 4. Compare with True Labels
# ------------------------
print("Predictions:")
for text, pred, true in zip(sample_sentiments, sample_pred_labels, true_labels):
    print(f"Text: {text[:50]}... | Predicted: {pred} | True: {true}")

# ------------------------
# 5. Evaluation Metrics
# ------------------------
from sklearn.metrics import classification_report, confusion_matrix

true_num = [ {'negative':0, 'neutral':1, 'positive':2}[label] for label in true_labels ]
pred_num = sample_pred_indices

print("\nClassification Report:")
print(classification_report(true_num, pred_num))

print("Confusion Matrix:")
print(confusion_matrix(true_num, pred_num))


## 6. Pre-trained Transformer Based Analysis
**Cons with Previous Approaches**:
- Limited training data
- Not context-aware
- Lack of semantic understanding

Pre-trained transformers (BERT, RoBERTa, etc...) addressed the cons above, because they are pre-trained with large amount of contents.

**Why Transformers?**
- Pre-trained models: You can leverage the knowledge from massive datasets it is already trained with.
- Deep context understandings: captures complex semantics.
- Flexible & Adaptable: can be fine tuned for specific domain, language or subset.
- It is multi-ligual: Models supporting multiple languages.
- Better handling of complex language components like negation, sarcasm, emojis, idioms, mixed reviews, etc...
- State-of-art performance and accuracy.
- Has specialized Tokenizer with [CLS] and [SEP]

**Steps followed with Transformers**
- Input text
- Tokenize
- Pass through pre-trained BERT
- Use the [CLS] token embedding
- Feed into a classifier (e.g. softmax layer)
- Output sentiment

**Tokenization with [CLS] and [SEP]**
[CLS] and [SEP] are special tokens used in BERT (and most Transformer-based models like RoBERTa).

[CLS] -> classification --> First token, used for sentence-level tasks (like sentiment)
[SEP] -> Separator --> End of sentence or separates sentence pairs

Examples:
```
Single sentence: "I love this product"
Tokenized: [CLS] I love this product [SEP]

Sentence pair: "I love this product" and "It works great"
Tokenized: [CLS] I love this product [SEP] It works great [SEP]
```

**Hugging Face** is a company and open-source community that focuses on natural language processing (NLP) and machine learning models, especially **transformers**.

**Variants of BERT**:
- DistilBERT: smaller, faster version of BERT. Optimized for sentiment analysis.
- RoBERTa: Robustly optimized BERT variant from Meta for sentiment analysis.
- mBERT: It is multi-lingual BERT. Supports around 100 languages.
- TwitterRoBERTa: Fine-tunned BERT for Twitter data.

In [ ]:
#Sample Reviews/posts
social_media_posts = [
    "I love this product! It's amazing!",
    "HPV is one of main reason for cancer.",
    "smoking Tobacco lead to cancer.",
    "I'm so happy with the results!",
    "Absolutely terrible, would not recommend.",
    "Me encanta este producto! Es increíble!",
    "C'est le pire service que j'ai jamais eu.",
    "この商品が大好きです！素晴らしい！",
    "이 제품을 정말 좋아해요! 놀라워요!",
    "இந்த தயாரிப்பு அருமை! மிகுந்த மகிழ்ச்சி!",
    "यह उत्पाद बहुत अच्छा है! यह अद्भुत है!",
    "Das humane Papillomavirus (HPV) kann Krebs verursachen.",
    "Le tabac est l'une des principales causes du cancer.",
    "La dieta equilibrata aiuta a prevenire molte malattie.",
    "Trop de soleil peut causer le cancer de la peau.",
    "Rauchen erhöht das Krebsrisiko erheblich.",
    "Una corretta alimentazione è essenziale per la salute.",
    "L'esposizione al sole senza protection peut être dangereuse.",
    "Il virus HPV è una delle principali cause di cancro cervicale.",
    "HPV ist eine ernsthafte Bedrohung für die Gesundheit von Frauen.",
    "Fumer nuit gravement à la santé et peut provoquer un cancer."
]

In [ ]:
!pip install langdetect

In [ ]:
from transformers import pipeline
import numpy as np

#Model1: Basic BERT, its only for English 'bert-base-uncased'
#note:Basic BERT is not optimized for Sentiment Analysis tasks

from langdetect import detect

english_posts=[post for post in social_media_posts if detect(post)=='en']

print(english_posts)

bert_analyzer=pipeline("sentiment-analysis",model="bert-base-uncased")
bert_result=bert_analyzer(english_posts)

bert_confidence=[result['score'] for result in bert_result]
print("BERT confidence Score:",bert_confidence)

#print labeled results with confidence
for post,result in zip(english_posts,bert_result):
    label=result['label']
    score=result['score']
    print(f"Post:{post}\n Sentiment Label:{label}\nConfidence Score:{score:.2f}\n")
    print("-"*60)

In [ ]:
# Model 2: DistilBERT is fine tuned for sentiment analaysis model name: "distillbert-base-uncased-finetuned-sst-2-english"
# Model is performing good on English but poorly on other languages

#Load the sentiment analyzer model pipeline
sent_analyzer=pipeline("sentiment-analysis",model="distilbert-base-uncased-finetuned-sst-2-english")
#Analyze sentiment for each post
sent_result=sent_analyzer(social_media_posts)
#Store and show confidence score and sentiment label
for post,result in zip(social_media_posts,sent_result):
    print(f"Post:{post}\n Sentiment Label:{result['label']}\nConfidence Score:{result['score']:.2f}\n")
    print('-'*50)

In [ ]:
#Model 3: RoBERTa : optimized BERT with improved training,better accuracy
#model name: 'roberta-base'

#Load the sentiment analyzer model pipeline
sent_analyzer=pipeline("sentiment-analysis",model="roberta-base")
#Analyze sentiment for each post
sent_result=sent_analyzer(social_media_posts)
#Store and show confidence score and sentiment label

label_mapping={'LABEL_0':'Negative','LABEL_1':'Positive'}

for post,result in zip(social_media_posts,sent_result):
    label_id=result['label']
    label_name=label_mapping.get(label_id,label_id)
    print(f"Post:{post}\n Sentiment Label:{label_name}\nConfidence Score:{result['score']:.2f}\n")
    print('-'*50)


In [ ]:
#Model 4: Sie RoBERTa : optimized BERT with improved training,better accuracy
#model name: 'siebert/sentiment-roberta-large-english '

#Load the sentiment analyzer model pipeline
sent_analyzer=pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")
#Analyze sentiment for each post
sent_result=sent_analyzer(social_media_posts)
#Store and show confidence score and sentiment label

label_mapping={'0':'Negative','1':'Positive'}

for post,result in zip(social_media_posts,sent_result):
    label_id=result['label']
    label_name=label_mapping.get(label_id,label_id)
    print(f"Post:{post}\n Sentiment Label:{label_name}\nConfidence Score:{result['score']:.2f}\n")
    print('-'*50)

In [ ]:
#Model 5: mBERT Multilingual BERT optimized for sentiment analysis
#model name:'nlptown/bert-base-multilingual-uncased-sentiment'
#Note: this model returns stars instead of the sentiment labels

#Load the sentiment analyzer model pipeline
sent_analyzer=pipeline("sentiment-analysis",model="nlptown/bert-base-multilingual-uncased-sentiment")
#Analyze sentiment for each post
sent_result=sent_analyzer(social_media_posts)
#Store and show confidence score and sentiment label

#Map stars to Sentiment labels
label_mapping={
    '1 star': 'Very Negative',
    '2 stars': 'Negative',
    '3 stars': 'Neutral',
    '4 stars': 'Positive',
    '5 stars': 'Very Positive'
}

pos_count=0
neg_count=0
for post,result in zip(social_media_posts,sent_result):
    label_id=result['label']
    label_name=label_mapping.get(label_id,label_id)
    if label_name=='Positive':
        pos_count+=1
    elif label_name=='Negative':
        neg_count+=1
    print(f"Post:{post}\n Sentiment Label:{label_name}\nConfidence Score:{result['score']:.2f}\n")
    print('-'*50)

print(f"No of positive posts:{pos_count}")
print(f"No of negative posts:{neg_count}")

#### Test with yelp data

In [ ]:
# ------------------------
# 1. Import Packages
# ------------------------
import pandas as pd
import re
import string
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# ------------------------
# 2. Load Dataset
# ------------------------
df = pd.read_csv("/content/drive/MyDrive/edurekaai/_data/yelp.csv", encoding="latin1")
df['text_cleaned'] = df['text'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', str(x).lower()))
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: re.sub(f"[{re.escape(string.punctuation)}]", '', x))
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: re.sub(r'\d+', '', x))

# ------------------------
# 3. Prepare Labels (Optional, for evaluation)
# ------------------------
df['sentiment'] = df['stars'].map(lambda x: 0 if x < 3 else (1 if x == 3 else 2))
label_map = {0: "negative", 1: "neutral", 2: "positive"}

# ------------------------
# 4. Load mBERT Model via Pipeline
# ------------------------
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"  # you can replace with a fine-tuned sentiment model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

sentiment_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True  # returns scores for all classes
)

# ------------------------
# 5. Predict Sentiment
# ------------------------
sample_texts = df['text_cleaned'][:15].tolist()  # pick first 5 reviews for demo

for text in sample_texts:
    result = sentiment_classifier(text)
    # Convert to label with highest score
    pred_label = max(result[0], key=lambda x: x['score'])['label']
    print(f"Text: {text[:50]}... -> Predicted Sentiment: {pred_label}")


**Note:**

Aspect Based Sentiment Analysis is on next file 07.